# Resume Merge — Interactive Notebook

Use this notebook to explore and debug the merge pipeline step by step.
Run cells in order from top to bottom.

In [ ]:
# Cell 1 — Imports and path setup
import sys, os, json, datetime
sys.path.insert(0, os.path.join('..', 'src'))

import fitz
import anthropic
from merge_resumes import PDFExtractor, ResumeComparator, HTMLBuilder, PDFRenderer

SOURCE_PDF    = os.path.join('..', 'input', 'source.pdf')
INPUT_PDF     = os.path.join('..', 'input', 'input.pdf')
TEMPLATE_HTML = os.path.join('..', 'templates', 'resume_merged.html')
OUTPUT_DIR    = os.path.join('..', 'output_files')

print('Imports OK')

In [ ]:
# Cell 2 — Extract text from source.pdf (destination/primary resume)
destination_text = PDFExtractor(SOURCE_PDF).extract_text()
print(f'Extracted {len(destination_text)} characters from source.pdf')
print('---')
print(destination_text[:2000])

In [ ]:
# Cell 3 — Extract text from input.pdf (donor/secondary resume)
donor_text = PDFExtractor(INPUT_PDF).extract_text()
print(f'Extracted {len(donor_text)} characters from input.pdf')
print('---')
print(donor_text[:2000])

In [ ]:
# Cell 4 — Side-by-side company name inspection (manual review before calling API)
import re

def extract_company_hints(text):
    # Very rough heuristic: lines that look like employer headings
    lines = text.splitlines()
    return [l.strip() for l in lines if len(l.strip()) > 5 and len(l.strip()) < 80
            and not l.strip().startswith('-') and not l.strip().startswith('•')]

dest_lines = extract_company_hints(destination_text)
donor_lines = extract_company_hints(donor_text)

print(f'--- Destination resume ({len(dest_lines)} candidate lines) ---')
for l in dest_lines[:40]:
    print(' ', l)

print(f'\n--- Donor resume ({len(donor_lines)} candidate lines) ---')
for l in donor_lines[:40]:
    print(' ', l)

In [ ]:
# Cell 5 — Run Claude comparison (costs API tokens)
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
comparator = ResumeComparator(client)
merge_data = comparator.compare(destination_text, donor_text)
print(json.dumps(merge_data, indent=2))

In [ ]:
# Cell 6 — Inspect missing jobs
missing_jobs = merge_data.get('missing_jobs', [])
print(f'{len(missing_jobs)} missing job(s):\n')
for j in missing_jobs:
    print(f"  {j.get('company')} | {j.get('dates')} | {j.get('title')}")

In [ ]:
# Cell 7 — Inspect missing skills
missing_skills = merge_data.get('missing_skills', [])
print(f'{len(missing_skills)} missing skill category(s):\n')
for s in missing_skills:
    items_preview = ', '.join(s.get('items', [])[:5])
    print(f"  [{s.get('category')}]: {items_preview}...")

In [ ]:
# Cell 8 — Load HTML template and build merged HTML
with open(TEMPLATE_HTML, encoding='utf-8') as f:
    template_html = f.read()

builder = HTMLBuilder(template_html)
merged_html = builder.build(merge_data)

print(f'Merged HTML length: {len(merged_html)} chars')
print('--- Preview (first 3000 chars) ---')
print(merged_html[:3000])

In [ ]:
# Cell 9 — Render PDF
os.makedirs(OUTPUT_DIR, exist_ok=True)
date_str = datetime.date.today().strftime('%Y-%m-%d')
output_path = os.path.join(OUTPUT_DIR, f'resume-merge-{date_str}.pdf')

PDFRenderer().render(merged_html, output_path)
print(f'PDF written to: {output_path}')

In [ ]:
# Cell 10 — Verify output
doc = fitz.open(output_path)
print(f'Pages: {doc.page_count}')
print(f'File size: {os.path.getsize(output_path):,} bytes')
print('--- First 500 chars of page 1 ---')
print(doc[0].get_text()[:500])